# Chapter 4: MCMC Retrievals for Non-Gaussian Posteriors

Optimal Estimation is powerful but assumes:
1. Gaussian prior
2. Gaussian noise
3. Posterior is Gaussian (only guaranteed if F is linear)

In real retrievals, these break down:
- **Aerosol retrievals**: AOD > 0 always → truncated prior
- **Cloud retrievals**: cloud fraction ∈ [0,1] → beta prior
- **Nonlinear forward models**: the posterior can be multimodal
- **Outlier measurements**: heavy-tailed noise (t-distribution better)

MCMC gives the **full posterior** without Gaussian assumptions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
np.random.seed(42)

try:
    import emcee
    import corner
    HAS_EMCEE = True
    print(f"emcee {emcee.__version__}")
except ImportError:
    HAS_EMCEE = False
    print("Install: pip install emcee corner")

# Physical constants
h = 6.626e-34; c_cgs = 3e10; kB = 1.38e-23; c2 = h*c_cgs/kB

def planck(nu, T):
    c1 = 2*h*c_cgs**2
    return c1*nu**3 / (np.expm1(c2*nu/T))

def rad_to_BT(nu, B):
    c1 = 2*h*c_cgs**2
    return c2*nu / np.log(1+c1*nu**3/B)

print("Ready.")


## 4.1 AOD Retrieval — Non-Gaussian Prior

**Problem:** Retrieve AOD at 550 nm and the Ångström exponent α from
reflectances at 3 MODIS-like bands.

The state vector: x = [AOD₅₅₀, α]

Physical constraints:
- AOD ≥ 0 (can't have negative extinction)
- α ∈ [0, 2.5] (physical range from particle size)

**Prior:**
- AOD ~ LogNormal (always positive, skewed)
- α ~ Uniform[0, 2.5]

**Forward model:** Simplified reflectance model R(λ) = f(AOD(λ), surface)


In [ ]:
# ── Simplified AOD forward model ──────────────────────────────────────────
wavelengths = np.array([470, 550, 860])   # nm (MODIS-like bands)

def AOD_spectral(AOD_550, alpha, wavelengths):
    return AOD_550 * (wavelengths / 550.0)**(-alpha)

def forward_model_AOD(theta, wavelengths, R_surface=0.05):
    AOD_550, alpha = theta
    AOD_wl  = AOD_spectral(AOD_550, alpha, wavelengths)
    T_atm   = np.exp(-AOD_wl / 0.6)   # simplified two-way transmittance
    R_path  = 0.1 * (1 - np.exp(-AOD_wl))   # path reflectance
    R_TOA   = R_path + T_atm**2 * R_surface / (1 - 0.07 * R_surface)
    return R_TOA

# True state
AOD_true   = 0.45
alpha_true = 1.3
R_surface  = 0.04   # dark ocean/vegetation

# Simulate measurement
R_obs_noiseless = forward_model_AOD([AOD_true, alpha_true], wavelengths, R_surface)
sigma_R  = 0.005   # reflectance noise
R_obs    = R_obs_noiseless + sigma_R * np.random.randn(3)

print(f"True state: AOD₅₅₀ = {AOD_true}, α = {alpha_true}")
print(f"Simulated reflectances: {np.round(R_obs, 4)}")

# ── Log-posterior ──────────────────────────────────────────────────────────
def log_prior(theta):
    AOD_550, alpha = theta
    if AOD_550 <= 0:       return -np.inf
    if not 0 < alpha < 2.5: return -np.inf
    # LogNormal prior on AOD: mode around 0.2, sigma_ln = 0.8
    lp_AOD   = stats.lognorm.logpdf(AOD_550, s=0.8, scale=np.exp(-0.3))
    # Uniform on alpha (already checked bounds)
    return lp_AOD

def log_likelihood(theta, R_obs, sigma_R, wavelengths, R_surface):
    AOD_550, alpha = theta
    if AOD_550 <= 0 or not (0 < alpha < 2.5):
        return -np.inf
    R_model = forward_model_AOD(theta, wavelengths, R_surface)
    return -0.5 * np.sum(((R_obs - R_model)/sigma_R)**2)

def log_posterior(theta, R_obs, sigma_R, wavelengths, R_surface):
    lp = log_prior(theta)
    if not np.isfinite(lp): return -np.inf
    return lp + log_likelihood(theta, R_obs, sigma_R, wavelengths, R_surface)

if HAS_EMCEE:
    ndim, nwalkers = 2, 32
    p0 = np.array([0.3, 1.0]) + 0.05*np.random.randn(nwalkers, ndim)
    p0[:, 0] = np.abs(p0[:, 0])   # ensure positive AOD

    sampler = emcee.EnsembleSampler(nwalkers, ndim, log_posterior,
                                     args=(R_obs, sigma_R, wavelengths, R_surface))
    sampler.run_mcmc(p0, 500, progress=True)
    sampler.reset()
    sampler.run_mcmc(None, 3000, progress=True)
    flat_samples = sampler.get_chain(flat=True)
    print(f"
Sampled {len(flat_samples)} posterior draws.")
else:
    # Synthetic samples for demo
    flat_samples = np.column_stack([
        np.abs(np.random.normal(AOD_true, 0.08, 10000)),
        np.random.normal(alpha_true, 0.2, 10000)
    ])
    flat_samples = flat_samples[(flat_samples[:,0]>0) & (flat_samples[:,1]>0)]
    print("Using synthetic samples.")


In [ ]:
# ── Visualise the AOD posterior ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Posterior on AOD_550
ax = axes[0]
ax.hist(flat_samples[:,0], bins=50, density=True,
        color="#3B8BD4", alpha=0.7, label="MCMC posterior")
ax.axvline(AOD_true, color="#D85A30", lw=2.5, ls="--", label=f"True AOD={AOD_true}")
ax.axvline(np.median(flat_samples[:,0]), color="#1D9E75", lw=2,
           label=f"Median={np.median(flat_samples[:,0]):.3f}")
# Prior
aod_x = np.linspace(0.001, 1.5, 200)
ax.plot(aod_x, stats.lognorm.pdf(aod_x, s=0.8, scale=np.exp(-0.3)),
        "gray", lw=2, ls=":", alpha=0.7, label="Prior (LogNormal)")
ax.set_xlabel("AOD at 550 nm"); ax.set_ylabel("Probability density")
ax.set_title("Marginal Posterior: AOD₅₅₀")
ax.legend(fontsize=8, frameon=False)

# Posterior on Ångström exponent
ax2 = axes[1]
ax2.hist(flat_samples[:,1], bins=50, density=True,
         color="#7F77DD", alpha=0.7, label="MCMC posterior")
ax2.axvline(alpha_true, color="#D85A30", lw=2.5, ls="--",
            label=f"True α={alpha_true}")
ax2.axvline(np.median(flat_samples[:,1]), color="#1D9E75", lw=2,
            label=f"Median={np.median(flat_samples[:,1]):.3f}")
ax2.set_xlabel("Ångström exponent α"); ax2.set_ylabel("Probability density")
ax2.set_title("Marginal Posterior: Ångström Exponent (fine vs coarse aerosol)")
ax2.legend(fontsize=8, frameon=False)

# Joint posterior + AOD spectrum
ax3 = axes[2]
ax3.scatter(flat_samples[::10, 0], flat_samples[::10, 1],
            alpha=0.15, s=4, color="#3B8BD4")
ax3.scatter(AOD_true, alpha_true, s=200, color="#D85A30", marker="*",
            zorder=5, label="True state")
ax3.scatter(np.median(flat_samples[:,0]), np.median(flat_samples[:,1]),
            s=200, color="#1D9E75", marker="o", zorder=5, label="Median")
ax3.set_xlabel("AOD at 550 nm")
ax3.set_ylabel("Ångström exponent α")
ax3.set_title("Joint Posterior P(AOD, α | R_obs) Correlation between aerosol parameters")
ax3.legend(fontsize=9, frameon=False)

plt.suptitle("MCMC Aerosol Retrieval — Full Non-Gaussian Posterior",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Credible intervals
for i, (name, unit) in enumerate([("AOD₅₅₀",""), ("α","")]):
    q = np.percentile(flat_samples[:,i], [5,16,50,84,95])
    print(f"{name}: {q[2]:.3f} +{q[3]-q[2]:.3f}/-{q[2]-q[1]:.3f}  (68% CI)  "
          f"[{q[0]:.3f}, {q[4]:.3f}] (90% CI)")


## 4.2 Posterior Predictive Check

After retrieval, simulate new reflectances from the posterior samples.
Do they look like the observations? This validates the forward model.


In [ ]:
# Posterior predictive check
n_ppc = 500
ppc_R = np.zeros((n_ppc, 3))
idx_ppc = np.random.choice(len(flat_samples), n_ppc)

for s, idx in enumerate(idx_ppc):
    theta_s = flat_samples[idx]
    R_s = forward_model_AOD(theta_s, wavelengths, R_surface)
    ppc_R[s] = R_s + sigma_R * np.random.randn(3)

band_names = ["470 nm
(Blue)", "550 nm
(Green)", "860 nm
(NIR)"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, (ax, bname) in enumerate(zip(axes, band_names)):
    ax.hist(ppc_R[:, i], bins=30, density=True, color="#3B8BD4",
            alpha=0.7, label="Posterior predictive")
    ax.axvline(R_obs[i], color="#D85A30", lw=2.5, ls="--",
               label=f"Observed R={R_obs[i]:.4f}")
    pB = np.mean(ppc_R[:, i] >= R_obs[i])
    ax.set_title(f"Band {bname}
p_B = {pB:.2f}")
    ax.set_xlabel("Reflectance")
    ax.legend(fontsize=8, frameon=False)

plt.suptitle("Posterior Predictive Check — Does the Model Reproduce the Observations?",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
print("p_B ≈ 0.5 for all bands → model fits well.")
print("p_B → 0 or 1 → model systematically over/under-predicts → consider model revision.")


In [ ]:
print("Chapter 4 complete!")
print()
print("When to use MCMC instead of OE:")
print("  - Prior is non-Gaussian (AOD: lognormal; cloud fraction: beta)")
print("  - Posterior is strongly non-Gaussian or multimodal")
print("  - State space is < ~20 dimensions (OE scales better in high-D)")
print("  - You need the full posterior for uncertainty propagation")
print()
print("MCMC retrieval pattern:")
print("  def log_prior(theta):  -> enforce physical bounds + prior shape")
print("  def log_likelihood(theta, y): -> radiative transfer misfit")
print("  def log_posterior(theta, y): -> sum of above")
print("  sampler = emcee.EnsembleSampler(nwalkers, ndim, log_posterior)")
